# Build / audit the Olympic geography crosswalk

This notebook combines the two approved canonical sources used by the map:

1. the Olympic source (`120_years_olympic_history_OG.csv`) tells us which delegation/NOC participated and its historical label;
2. CShapes 2.0 tells us which sovereign-state geometries and Gleditsch–Ward codes exist in each edition.

The output `olympic_geography_mapping.csv` is **project-authored configuration**, not a new external dataset. Straightforward unique name matches can be suggested automatically; historical/composite/non-sovereign cases require explicit review. The committed crosswalk is then validated against every Summer Olympic `NOC × edition` from 1952–1988.

In [ ]:
from pathlib import Path
import re
import unicodedata
import pandas as pd

HERE = Path.cwd()
if HERE.name != 'geography':
    candidates = [x / 'preprocessing' / 'source' / 'geography' for x in [Path.cwd(), *Path.cwd().parents]]
    HERE = next((x for x in candidates if (x / 'SOURCE.md').exists()), None)
    if HERE is None:
        raise RuntimeError('Run from the repository or preprocessing/source/geography directory.')

PRE = HERE.parents[1]
OLYMPIC = HERE.parent / 'olympics' / '120_years_olympic_history_OG.csv'
CSHAPES_REFERENCE = PRE / 'intermediate' / 'cshapes_state_reference.csv'
MAPPING = HERE / 'olympic_geography_mapping.csv'
REVIEW = PRE / 'intermediate' / 'olympic_geography_review.csv'
YEARS = [1952, 1956, 1960, 1964, 1968, 1972, 1976, 1980, 1984, 1988]

if not CSHAPES_REFERENCE.exists():
    raise FileNotFoundError(
        'Missing preprocessing/intermediate/cshapes_state_reference.csv. '
        'Run generate_cshapes_snapshots.R first when intentionally rebuilding the crosswalk.'
    )

## 1. Build the review worksheet

The Olympic label is taken from the most frequent country/delegation-like `Team` label across the full Cold War scope, not from a single edition. This avoids sailing boat names such as `Yeoman` becoming country names. The name matcher is deliberately conservative: it suggests only a unique CShapes state-name match. Anything else stays unresolved for explicit review.

In [ ]:
def normalize_name(value):
    text = unicodedata.normalize('NFKD', str(value))
    text = ''.join(ch for ch in text if not unicodedata.combining(ch)).casefold()
    return re.sub(r'[^a-z0-9]+', ' ', text).strip()

raw = pd.read_csv(OLYMPIC, delimiter=';', low_memory=False)
scope = raw[(raw.Season == 'Summer') & raw.Year.isin(YEARS)].copy()
scope['TeamClean'] = scope.Team.astype(str).str.replace(r'-\d+$', '', regex=True).str.strip()
labels = (
    scope.groupby('NOC').TeamClean
    .agg(lambda s: s.value_counts().index[0])
    .rename('OlympicLabel')
)
participants = scope[['Year', 'NOC']].drop_duplicates().merge(labels, on='NOC', how='left')

cshapes = pd.read_csv(CSHAPES_REFERENCE)
cshapes = cshapes[cshapes.Year.isin(YEARS)].copy()
cshapes['NameKey'] = cshapes.CShapesName.map(normalize_name)
participants['NameKey'] = participants.OlympicLabel.map(normalize_name)

candidate = participants.merge(
    cshapes[['Year', 'NameKey', 'CShapesName', 'GwCode']],
    on=['Year', 'NameKey'], how='left'
)
counts = candidate.groupby(['Year', 'NOC']).GwCode.transform(lambda s: s.notna().sum())
candidate['CandidateStatus'] = counts.map({1: 'unique_name_match'}).fillna('needs_review')
candidate.to_csv(REVIEW, index=False)
print(f'Wrote review worksheet: {REVIEW}')
candidate.CandidateStatus.value_counts()

## 2. Validate the reviewed crosswalk

The committed CSV contains the reviewed decision for every delegation. Historical names are preserved, composite mappings may contain multiple GW codes, and deliberately unmapped delegations carry an explicit exclusion reason. No guessing/fallback is allowed during normal builds.

In [ ]:
mapping = pd.read_csv(MAPPING, dtype={'GwCodes': str}).fillna({'GwCodes': '', 'Reason': ''})

resolved = []
for row in participants.itertuples(index=False):
    match = mapping[
        mapping.NOC.eq(row.NOC)
        & mapping.StartYear.le(int(row.Year))
        & mapping.EndYear.ge(int(row.Year))
    ]
    if len(match) != 1:
        raise ValueError(f'Expected one reviewed mapping for {row.NOC} {row.Year}; found {len(match)}')
    m = match.iloc[0]
    resolved.append({
        'Year': int(row.Year), 'NOC': row.NOC, 'OlympicLabel': row.OlympicLabel,
        'Country': m.Country, 'GwCodes': m.GwCodes, 'Status': m.Status, 'Reason': m.Reason,
    })
resolved = pd.DataFrame(resolved)
resolved.head()

In [ ]:
# Every mapped GW code must exist in that edition's CShapes reference.
valid_codes = {
    int(year): set(group.GwCode.astype(int))
    for year, group in cshapes.groupby('Year')
}
owners = {}
for row in resolved.itertuples(index=False):
    codes = [int(x) for x in str(row.GwCodes).split(';') if x]
    if row.Status == 'mapped' and not codes:
        raise ValueError(f'Mapped row has no GW code: {row.NOC} {row.Year}')
    if row.Status != 'mapped' and not str(row.Reason).strip():
        raise ValueError(f'Excluded row has no reason: {row.NOC} {row.Year}')
    for code in codes:
        if code not in valid_codes[row.Year]:
            raise ValueError(f'GW {code} missing in CShapes {row.Year}: {row.NOC}')
        key = (row.Year, code)
        if key in owners and owners[key] != row.NOC:
            raise ValueError(f'GW collision {key}: {owners[key]} and {row.NOC}')
        owners[key] = row.NOC

print(f'Validated {resolved.NOC.nunique()} NOCs and {len(resolved)} NOC×edition rows.')

The normal website/data build uses the committed crosswalk directly and therefore does not require R. This notebook is only needed when the historical geography source or matching decisions are intentionally regenerated/re-audited.